In [ ]:
# here is how we activate an environment in our current directory
# 这里演示如何激活当前目录下的 Julia 环境
import Pkg; Pkg.activate(@__DIR__)

# instantate this environment (download packages if you haven't)
# 实例化该环境（下载尚未安装的依赖包）
Pkg.instantiate();

# let's load LinearAlgebra in
# 加载 LinearAlgebra 标准库
using LinearAlgebra
using Test

# Question 1: Differentiation(微分) in Julia (10 pts)
Julia has a fast and easy to use forward-mode automatic differentiation package called [ForwardDiff.jl](https://github.com/JuliaDiff/ForwardDiff.jl) that we will make use of throughout this course. In general it is easy to use and very fast, but there are a few quirks that are detailed below. This notebook will start by walking through general usage for the following cases:
- functions with a single input 
- functions with multiple inputs
- composite functions

as well as a guide on how to avoid the most common ForwardDiff.jl error caused by creating arrays inside the function being differentiated. First, let's look at the ForwardDiff.jl functions that we are going to use:
- `FD.derivative(f,x)` derivative of scalar or vector valued f wrt scalar x 
- `FD.jacobian(f,x)` jacobian of vector valued f wrt vector x
- `FD.gradient(f,x)` gradient of scalar valued f wrt vector x 
- `FD.hessian(f,x)` hessian of scalar valued f wrt vector x 
---
[ForwardDiff.jl] 是 Julia 生态中一个快速且易用的前向模式自动微分包，本课程中将经常使用。它总体上易于使用且速度极快，但存在一些需要留意的细节。下表总结了常见的用法及其注意事项:
-单输入函数
-多输入函数
-复合函数

此外，这里还提供一份指南，教你如何避免 ForwardDiff.jl 中最常见的错误——即在被微分的函数内部创建数组所引发的问题。首先，我们来了解一下将会用到的 ForwardDiff.jl 函数：
- `FD.derivative(f,x)` 函数$f$可为标量值或向量值，对自变量 $x$（标量）求导。
- `FD.jacobian(f,x)` 函数$f$为向量值，关于向量 $x$ 求雅可比矩阵。
- `FD.gradient(f,x)` 函数$f$为标量值，关于向量 $x$ 求梯度。
- `FD.hessian(f,x)` 函数$f$为标量值，关于向量 $x$ 求海森矩阵。

### Note on gradients:  关于梯度的注意事项
For an arbitrary function $f(x):\mathbb{R}^N \rightarrow \mathbb{R}^M$, the jacobian is the following:
对于任意函数 $f(x):\mathbb{R}^N \rightarrow \mathbb{R}^M$其雅可比矩阵定义如下：

$$\frac{\partial f(x)}{\partial x} = \left[\begin{array}{ccc}
\frac{\partial f_{1}}{\partial x_{1}} & \cdots & \frac{\partial f_{1}}{\partial x_{n}} \\
\vdots & \ddots & \vdots \\
\frac{\partial f_{m}}{\partial x_{1}} & \cdots & \frac{\partial f_{m}}{\partial x_{n}}
\end{array}\right] $$


Now if we have a scalar valued function (like a cost function) $f(x):\mathbb{R}^N \rightarrow \mathbb{R}$, the jacobian is the following row vector:
现在，如果考虑一个标量值函数（例如代价函数）$f(x):\mathbb{R}^N \rightarrow \mathbb{R}$，其雅可比矩阵是一个行向量，形式如下：

$$\frac{\partial f(x)}{\partial x} = \left[\begin{array}{ccc}
\frac{\partial f_{1}}{\partial x_{1}} & \cdots & \frac{\partial f_{1}}{\partial x_{n}}
\end{array}\right] $$

The transpose of this jacobian for scalar valued functions is called the gradient:
对标量值函数而言，该雅可比矩阵的转置被称为梯度：

$$ \nabla f(x) = \bigg[\frac{\partial f(x)}{\partial x}\bigg]^T $$

TLDR:
TLDR（太长不读）：
- the jacobian of a scalar value function is a row vector 
- 标量值函数的雅可比矩阵是一个行向量
- the gradient is the transpose of this jacobian, making the gradient a column vector 
- 梯度是该雅可比矩阵的转置，因此梯度是一个列向量
- ForwardDiff.jl will give you an error if you try to take a jacobian of a scalar valued function, use the gradient function instead
- ForwardDiff.jl 对标量值函数求雅可比矩阵会报错，请改用 gradient 函数

## Part (a): General usage (2 pts)
The API for functions with one input is detailed below:
---
## 第 (a) 部分：一般用法（2 分）
下面详细给出单输入函数对应的 API：


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out.
# 注意：此代码块是教程，不需要填写任何内容。

#---------load the package-----------
# ---------加载包-----------
# using ForwardDiff # this puts all exported functions into our namespace 
# 用 using 导入时，所有导出的函数会直接进入当前命名空间
# import ForwardDiff # this means we have to use ForwardDiff.<function name>
# 用 import 导入时，必须写成 ForwardDiff.<函数名> 的形式
import ForwardDiff as FD # this let's us do FD.<function name> 
                         # 这样我们就能用 FD.<函数名> 来调用

function foo1(x)
    #scalar input, scalar output
    # 标量输入，标量输出
    return sin(x)*cos(x)^2
end

function foo2(x)
    # vector input, scalar output
    # 向量输入，标量输出
    return sin(x[1]) + cos(x[2])
end
function foo3(x)
    # vector input, vector output
    # 向量输入，向量输出
    return [sin(x[1])*x[2];cos(x[2])*x[1]]
end


let # we just use this to avoid creating global variables
    # 用 let 块避免创建全局变量
    
    # evaluate the derivative of foo1 at x1
    # 在 x1 处计算 foo1 的导数
    x1 = 5*randn();
    @show ∂foo1_∂x = FD.derivative(foo1, x1);
    
    # evaluate the gradient and hessian of foo2 at x2
    # 在 x2 处计算 foo2 的梯度和海森矩阵
    x2 = 5*randn(2);
    @show ∇foo2 = FD.gradient(foo2, x2);
    @show ∇²foo2 = FD.hessian(foo2, x2);
    
    # evluate the jacobian of foo3 at x2
    # 在 x2 处计算 foo3 的雅可比矩阵
    @show ∂foo3_∂x = FD.jacobian(foo3,x2);
    
end

In [ ]:
# here is our function of interest
# 这是我们关心的函数
function foo4(x)
    Q = diagm([1;2;3.0]) # this creates a diagonal matrix from a vector
                         # 从一个向量构造对角矩阵
    return 0.5*x'*Q*x/x[1] - log(x[1])*exp(x[2])^x[3] 
end

function foo4_expansion(x)
    # TODO: this function should output the hessian H and gradient g of the function foo4
    # TODO：该函数应输出 foo4 的海森矩阵 H 和梯度 g
    
    # TODO: calculate the gradient of foo4 evaluated at x
    # TODO：计算 foo4 在 x 处的梯度
    g = zeros(length(x))
    
    # TODO: calculate the hessian of foo4 evaluated at x
    # TODO：计算 foo4 在 x 处的海森矩阵
    H = zeros(length(x),length(x))
    
    return g, H
end

In [ ]:
@testset "1a" begin                        
    x = [.2;.4;.5]
    g,H = foo4_expansion(x)
    @test isapprox(g,[-18.98201379080085, 4.982885952667278, 8.286308762133823],atol = 1e-8)        
    @test norm(H -[164.2850689540042 -23.053506895400425 -39.942805516320334;
                             -23.053506895400425 10.491442976333639 2.3589262864014673;
                             -39.94280551632034 2.3589262864014673 15.314523504853529]) < 1e-8 
end

## Part (b): Derivatives for functions with multiple input arguments (2 pts)
---
## 第 (b) 部分：多输入参数函数的导数（2 分）


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out. 
# 注意：此代码块是教程，不需要填写任何内容。

# calculate derivatives for functions with multiple inputs 
# 计算多输入函数的导数
function dynamics(x,a,b,c)
    return [x[1]*a; b*c*x[2]*x[1]]
end

let 
    x1 = randn(2)
    a = randn()
    b = randn()
    c = randn()
    
    # this evaluates the jacobian with respect to x, given a, b, and c
    # 给定 a、b、c，计算关于 x 的雅可比矩阵
    A1 = FD.jacobian(dx -> dynamics(dx, a, b, c), x1)
    
    # it doesn't matter what we call the new variable
    # 新变量叫什么名字并不重要
    A2 = FD.jacobian(_x -> dynamics(_x, a, b, c), x1)
    
    # alternatively we can do it like this using a closure
    # 另一种做法：利用闭包（closure）
    dynamics_just_x(_x) = dynamics(_x, a, b, c) 
    A3 = FD.jacobian(dynamics_just_x, x1)
    
    @test norm(A1 - A2) < 1e-13 
    @test norm(A1 - A3) < 1e-13
end

In [ ]:
function eulers(x,u,J)
    # dynamics when x is angular velocity and u is an input torque
    # 动力学：x 是角速度，u 是输入力矩
    ẋ = J\(u - cross(x,J*x))
    return ẋ
end

function eulers_jacobians(x,u,J)
    # given x, u, and J, calculate the following two jacobians 
    # 给定 x、u 和 J，计算下面两个雅可比矩阵
    
    # TODO: fill in the following two jacobians
    # TODO：填写下面两个雅可比矩阵
    
    # ∂ẋ/∂x
    # ∂ẋ/∂x：状态导数对状态 x 的雅可比矩阵
    A = zeros(3,3)
    
    # ∂ẋ/∂u
    # ∂ẋ/∂u：状态导数对输入 u 的雅可比矩阵
    B = zeros(3,3)
    
    return A, B
end

In [ ]:
@testset "1b" begin                                                
    
    x = [.2;-7;.2]
    u = [.1;-.2;.343]
    J = diagm([1.03;4;3.45])
    
    A,B = eulers_jacobians(x,u,J)

    skew(v) = [0 -v[3] v[2]; v[3] 0 -v[1]; -v[2] v[1] 0]
    @test isapprox(A,-J\(skew(x)*J - skew(J*x)), atol = 1e-8)  

    @test norm(B - inv(J)) < 1e-8                

end

## Part (c): Derivatives of composite functions (1 pts)
---
## 第 (c) 部分：复合函数的导数（1 分）


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out. 
# 注意：此代码块是教程，不需要填写任何内容。
function f(x)
    return x[1]*x[2]
end
function g(x)
    return [x[1]^2; x[2]^3]
end

let 
    x1 = 2*randn(2)
    
    # using gradient of the composite function
    # 对复合函数直接使用梯度
    ∇f_1 = FD.gradient(dx -> f(g(dx)), x1)
    
    # using the chain rule 
    # 使用链式法则
    J = FD.jacobian(g, x1)
    ∇f_2 = J'*FD.gradient(f, g(x1))
    
    @show norm(∇f_1 - ∇f_2)
end

In [ ]:
function f2(x)
    return x*sin(x)/2
end
function g2(x)
    return cos(x)^2 - tan(x)^3
end

function composite_derivs(x)
    
    # TODO: return ∂y/∂x where y = g2(f2(x)) 
    # TODO：返回 ∂y/∂x，其中 y = g2(f2(x))
    # (hint: this is 1D input and 1D output, so it's ForwardDiff.derivative)
    # （提示：一维输入、一维输出，因此使用 ForwardDiff.derivative）
    return 0.0
end    

In [ ]:
@testset "1c" begin                                           
    x = 1.34 
    deriv = composite_derivs(x)

    @test isapprox(deriv,-2.390628273373545,atol = 1e-8)  
end

## Part (d): Fixing the most common ForwardDiff error (2 pt)
First we will show an example of this error:
---
## 第 (d) 部分：修复最常见的 ForwardDiff 报错（2 分）
首先，我们展示一个这种错误的例子：


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out. 
# 注意：此代码块是教程，不需要填写任何内容。
function f_zero_1(x)
    println("-------types of input x---------")
    @show typeof(x) # print out type of x 
                    # 打印出 x 的类型
    @show eltype(x) # print out the element type of x 
                    # 打印出 x 的元素类型
    
    
    xdot = zeros(length(x)) # this default creates zeros of type Float64
                            # 默认创建 Float64 类型的零向量
    println("-------types of output xdot---------")
    @show typeof(xdot)
    @show eltype(xdot)
    
    # these lines will error because i'm trying to put a ForwardDiff.dual 
    # 这些行会报错，因为试图把 ForwardDiff.Dual
    # inside of a Vector{Float64}
    # 放进 Vector{Float64} 中
    xdot[1] = x[1]*x[2]
    xdot[2] = x[2]^2
    
    return xdot 
end

let 
    # try and calculate the jacobian of f_zero_1 on x1
    # 尝试计算 f_zero_1 在 x1 处的雅可比矩阵
    x1 = randn(2)
    @info "this error is expected:"
    try 
        FD.jacobian(f_zero_1,x1)
    catch e 
        buf = IOBuffer()
        showerror(buf,e)
        message = String(take!(buf))
        Base.showerror(stdout,e)
    end
end

This is the most common ForwardDiff error that you will encounter. ForwardDiff works by pushing `ForwardDiff.Dual` variables through the function being differentiated. Normally this works without issue, but if you create a vector of `Float64` (like you would with `xdot = zeros(5)`, it is unable to fit the `ForwardDiff.Dual`'s in with the `Float64`'s. To get around this, you have two options:
---
这是你经常会遇到的最常见的 ForwardDiff 报错。ForwardDiff 的原理是把 `ForwardDiff.Dual` 变量穿过被求导的函数。
正常情况下没有问题，但如果你创建了一个 `Float64` 向量（例如 `xdot = zeros(5)`），它无法把 `ForwardDiff.Dual` 放进 `Float64` 数组里。
要绕过这个问题，你有两种选择：


### Option 1 
Our first option is just creating xdot directly, without creating an array of zeros to index into. 
---
### 方案 1
第一种方案是直接构造 xdot，而不是先创建一个零数组再去索引赋值。


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out. 
# 注意：此代码块是教程，不需要填写任何内容。
function f_zero_1(x)
    
    # let's create xdot directly, without first making a vector of zeros 
    # 直接构造 xdot，而不是先创建零向量
    xdot = [x[1]*x[2], x[2]^2]
    
    # NOTE: the compiler figures out which type to make xdot, so when you call the function normally
    # 注意：编译器会自动推断 xdot 的类型；正常调用时
    # it's a Float64, and when it's being diffed, it's automatically promoted to a ForwardDiff.Dual type
    # 它是 Float64；被求导时会自动提升为 ForwardDiff.Dual 类型
    
    println("-------types of input x---------")
    @show typeof(x) # print out type of x 
                    # 打印出 x 的类型
    @show eltype(x) # print out the element type of x 
                    # 打印出 x 的元素类型
    
    println("-------types of output xdot---------")
    @show typeof(xdot)
    @show eltype(xdot)
    
    return xdot 
end

let 
    # try and calculate the jacobian of f_zero_1 on x1
    # 尝试计算 f_zero_1 在 x1 处的雅可比矩阵
    x1 = randn(2)
    FD.jacobian(f_zero_1,x1) # this will work
                             # 这样就能正常工作
end

### Option 2
The second option is to create the array of zeros in a way that accounts for the input type. This can be done by replacing `zeros(length(x))` with `zeros(eltype(x),length(x))`. The first argument `eltype(x)` simply creates a vector of zeros that is the same type as the element type in vector x. 
---
### 方案 2
第二种方案是按照输入类型来创建零数组：把 `zeros(length(x))` 换成 `zeros(eltype(x),length(x))`。
第一个参数 `eltype(x)` 让零向量与 x 的元素类型保持一致。


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out. 
# 注意：此代码块是教程，不需要填写任何内容。
function f_zero_1(x)
    
    xdot = zeros(eltype(x), length(x))
    
    xdot[1] = x[1]*x[2]
    xdot[2] = x[2]^2
    
    println("-------types of input x---------")
    @show typeof(x) # print out type of x 
                    # 打印出 x 的类型
    @show eltype(x) # print out the element type of x 
                    # 打印出 x 的元素类型
    
    println("-------types of output xdot---------")
    @show typeof(xdot)
    @show eltype(xdot)
    
    return xdot 
end

let 
    # try and calculate the jacobian of f_zero_1 on x1
    # 尝试计算 f_zero_1 在 x1 处的雅可比矩阵
    x1 = randn(2)
    FD.jacobian(f_zero_1,x1) # this will fail! 
                             # 这样会失败！
end

Now you can show that you understand these two options by fixing two broken functions.
---
现在通过修复两个有问题的函数，证明你理解了这两种方案。


In [ ]:
# TODO: fix this error when trying to diff through this function
# TODO：修复对这个函数求导时的类型错误
# hint: you can use promote_type(eltype(x),eltype(u)) to return the correct type if either x or u is a ForwardDiff.Dual (option 1)
# 提示：若 x 或 u 是 ForwardDiff.Dual，可用 promote_type(eltype(x),eltype(u)) 得到正确类型（方案 1）

function dynamics(x,u)
    xdot = zeros(length(x))
    xdot[1] = x[1]*sin(u[1])
    xdot[2] = x[2]*cos(u[2])
    return xdot
end

In [ ]:
@testset "1d" begin                                     
    x = [.1;.4]
    u = [.2;-.3]
    A = FD.jacobian(_x -> dynamics(_x,u),x) 
    B = FD.jacobian(_u -> dynamics(x,_u),u) 
    @test typeof(A) == Matrix{Float64}                  
    @test typeof(B) == Matrix{Float64}                  
end

## Finite Difference Derivatives 
If you ever have trouble working through a ForwardDiff error, you should always feel free to use the [FiniteDiff.jl](https://github.com/JuliaDiff/FiniteDiff.jl) FiniteDiff.jl package instead. This computes derivatives through a [finite difference method](https://en.wikipedia.org/wiki/Finite_difference_method). This is slower and less accurate than ForwardDiff, but it will always work so long as the function works.  

Before with ForwardDiff we had this:

- `FD.derivative(f,x)` derivative of scalar or vector valued f wrt scalar x 
- `FD.jacobian(f,x)` jacobian of vector valued f wrt vector x
- `FD.gradient(f,x)` gradient of scalar valued f wrt vector x 
- `FD.hessian(f,x)` hessian of scalar valued f wrt vector x 

Now with FiniteDiff we have this:

- `FD2.finite_difference_derivative(f,x)` derivative of scalar or vector valued f wrt scalar x 
- `FD2.finite_difference_jacobian(f,x)` jacobian of vector valued f wrt vector x
- `FD2.finite_difference_gradient(f,x)` gradient of scalar valued f wrt vector x 
- `FD2.finite_difference_hessian(f,x)` hessian of scalar valued f wrt vector x 
---
## 有限差分导数
如果你在使用 ForwardDiff 时遇到难以解决的报错，可以改用 FiniteDiff.jl 包。它通过有限差分方法计算导数。
它比 ForwardDiff 更慢、精度更低，但只要函数本身能运行，它通常都能工作。

之前用 ForwardDiff 我们有：
- `FD.derivative(f,x)`：对标量或向量值函数 f 关于标量 x 求导数
- `FD.jacobian(f,x)`：对向量值函数 f 关于向量 x 求雅可比矩阵
- `FD.gradient(f,x)`：对标量值函数 f 关于向量 x 求梯度
- `FD.hessian(f,x)`：对标量值函数 f 关于向量 x 求海森矩阵

现在换成 FiniteDiff 我们有：
- `FD2.finite_difference_derivative(f,x)`：有限差分版导数
- `FD2.finite_difference_jacobian(f,x)`：有限差分版雅可比矩阵
- `FD2.finite_difference_gradient(f,x)`：有限差分版梯度
- `FD2.finite_difference_hessian(f,x)`：有限差分版海森矩阵


In [ ]:
# NOTE: this block is a tutorial, you do not have to fill anything out.
# 注意：此代码块是教程，不需要填写任何内容。

# load the package 
# 加载该包
import FiniteDiff as FD2 

function foo1(x)
    #scalar input, scalar output
    # 标量输入，标量输出
    return sin(x)*cos(x)^2
end

function foo2(x)
    # vector input, scalar output
    # 向量输入，标量输出
    return sin(x[1]) + cos(x[2])
end
function foo3(x)
    # vector input, vector output
    # 向量输入，向量输出
    return [sin(x[1])*x[2];cos(x[2])*x[1]]
end


let # we just use this to avoid creating global variables
    # 用 let 块避免创建全局变量
    
    # evaluate the derivative of foo1 at x1
    # 在 x1 处计算 foo1 的导数
    x1 = 5*randn();
    @show ∂foo1_∂x = FD2.finite_difference_derivative(foo1, x1);
    
    # evaluate the gradient and hessian of foo2 at x2
    # 在 x2 处计算 foo2 的梯度和海森矩阵
    x2 = 5*randn(2);
    @show ∇foo2 = FD2.finite_difference_gradient(foo2, x2);
    @show ∇²foo2 = FD2.finite_difference_hessian(foo2, x2);
    
    # evluate the jacobian of foo3 at x2
    # 在 x2 处计算 foo3 的雅可比矩阵
    @show ∂foo3_∂x = FD2.finite_difference_jacobian(foo3,x2);
    
    @test norm(∂foo1_∂x - FD.derivative(foo1, x1)) < 1e-4 
    @test norm(∇foo2 - FD.gradient(foo2, x2)) < 1e-4 
    @test norm(∇²foo2 - FD.hessian(foo2, x2)) < 1e-4 
    @test norm(∂foo3_∂x - FD.jacobian(foo3, x2)) < 1e-4 
    
    
end